In [1]:
pip install torch transformers pandas scikit-learn matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import time
import gc
from tqdm import tqdm
import os
import platform

# ==========================================
# 0. M4 Pro 硬體加速設定
# ==========================================
# 設定繪圖字型
system_name = platform.system()
if system_name == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC', 'PingFang TC']
else:
    plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

# 檢測 MPS 加速
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 M4 Pro (MPS) 效能全開模式已啟動！")
    # 針對 M系列晶片的環境變數優化 (可選，但在程式碼中提示)
    os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" 
else:
    device = torch.device("cpu")
    print("⚠️ 未偵測到 MPS，將使用 CPU (速度會慢很多)。")

# ==========================================
# 1. 參數設定 (Config)
# ==========================================
TRAIN_PKL = 'train_seg.pkl'
TEST_PKL = 'test_seg.pkl'

# Facebook 官方多語言模型
MODEL_NAME = 'xlm-roberta-base' 

MAX_LEN = 256
BATCH_SIZE = 32     # 🔥 M4 Pro 24GB 建議設 32，跑滿效能
EPOCHS = 3
LEARNING_RATE = 2e-5
K_FOLDS = 10

# ==========================================
# 2. 資料準備 (Dataset)
# ==========================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer.encode_plus(
            text, 
            add_special_tokens=True, 
            max_length=self.max_len,
            padding='max_length', 
            truncation=True, 
            return_attention_mask=True, 
            return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = NewsDataset(
        texts=df['text_clean'].to_numpy(),
        labels=df['label_idx'].to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    # 🔥 pin_memory=True 對 M4 Unified Memory 傳輸有幫助
    # 🔥 num_workers=0 是 Mac 穩定跑最快的設定
    return DataLoader(ds, batch_size=batch_size, num_workers=0, pin_memory=True)

# ==========================================
# 3. 訓練與評估核心函式
# ==========================================
def train_epoch(model, data_loader, optimizer, scheduler, device, epoch_idx):
    model = model.train()
    losses = []
    correct_predictions = 0
    total_samples = 0
    
    loop = tqdm(data_loader, leave=False, desc=f"   Train Epoch {epoch_idx}")
    
    for d in loop:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_samples += labels.size(0)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())

    return correct_predictions.float() / total_samples, np.mean(losses)

def eval_model(model, data_loader, device):
    model = model.eval()
    predictions = []
    real_values = []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            real_values.extend(labels.cpu().tolist())
            
    return accuracy_score(real_values, predictions)

# ==========================================
# 主程式
# ==========================================
if __name__ == '__main__':
    print("1. 讀取與處理資料...")
    if not os.path.exists(TRAIN_PKL) or not os.path.exists(TEST_PKL):
        print(f"❌ 找不到 {TRAIN_PKL} 或 {TEST_PKL}，請確認檔案路徑！")
    else:
        train_df = pd.read_pickle(TRAIN_PKL)
        test_df = pd.read_pickle(TEST_PKL)
        
        # XLM-R 轉字串防呆
        train_df['text_clean'] = train_df['text'].fillna('').apply(lambda x: str(x))
        test_df['text_clean'] = test_df['text'].fillna('').apply(lambda x: str(x))
        
        le = LabelEncoder()
        train_df['label_idx'] = le.fit_transform(train_df['label'])
        test_df['label_idx'] = le.transform(test_df['label'])
        label_names = le.classes_
        num_classes = len(label_names)
        
        print(f"正在載入 Facebook AI 模型: {MODEL_NAME} ...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        # --- 2. 執行 10-Fold Cross Validation ---
        print(f"\n2. 開始 {K_FOLDS}-Fold Cross Validation...")
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        
        cv_accuracies = []
        fold_start_total = time.time()

        X = train_df
        y = train_df['label_idx']

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            print(f"\n🔹 Fold {fold + 1}/{K_FOLDS}")
            
            fold_train = X.iloc[train_idx]
            fold_val = X.iloc[val_idx]
            
            # 使用 Pin Memory 加速
            train_loader = create_data_loader(fold_train, tokenizer, MAX_LEN, BATCH_SIZE)
            val_loader = create_data_loader(fold_val, tokenizer, MAX_LEN, BATCH_SIZE)
            
            model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
            model = model.to(device)
            
            optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
            total_steps = len(train_loader) * EPOCHS
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
            
            for epoch in range(EPOCHS):
                train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, epoch+1)
            
            val_acc = eval_model(model, val_loader, device)
            cv_accuracies.append(val_acc)
            print(f"   ✅ Fold {fold+1} Val Accuracy: {val_acc:.4f}")
            
            # 🔥 M4 Pro 記憶體釋放術
            del model, optimizer, scheduler
            if torch.backends.mps.is_available(): 
                torch.mps.empty_cache() # 強制釋放 MPS 記憶體
            elif torch.cuda.is_available(): 
                torch.cuda.empty_cache()
            gc.collect()

        avg_cv_acc = np.mean(cv_accuracies)
        print(f"\n🏆 {K_FOLDS}-Fold CV 平均準確率: {avg_cv_acc:.4f}")

        # --- 3. 最終全量訓練 ---
        print(f"\n3. 使用完整訓練集進行最終模型訓練 (Epochs: {EPOCHS})...")
        final_start = time.time()
        
        full_train_loader = create_data_loader(train_df, tokenizer, MAX_LEN, BATCH_SIZE)
        test_loader = create_data_loader(test_df, tokenizer, MAX_LEN, BATCH_SIZE)
        
        final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
        final_model = final_model.to(device)
        optimizer = AdamW(final_model.parameters(), lr=LEARNING_RATE)
        total_steps = len(full_train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        
        for epoch in range(EPOCHS):
            print(f"   Final Epoch {epoch+1}/{EPOCHS}")
            train_acc, train_loss = train_epoch(final_model, full_train_loader, optimizer, scheduler, device, epoch+1)
            print(f"   -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
            
        final_train_time = time.time() - final_start

        # --- 4. 評估 ---
        print("\n4. 最終測試集評估...")
        final_model.eval()
        predictions, probs_list, real_values = [], [], []
        
        with torch.no_grad():
            for d in tqdm(test_loader, desc="Testing"):
                input_ids = d["input_ids"].to(device)
                attention_mask = d["attention_mask"].to(device)
                labels = d["labels"].to(device)

                outputs = final_model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)

                predictions.extend(preds.cpu().tolist())
                probs_list.extend(probs.cpu().tolist())
                real_values.extend(labels.cpu().tolist())

        y_true = np.array(real_values)
        y_pred = np.array(predictions)
        
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
        
        print("\n" + "="*80)
        print("🏆 Facebook XLM-RoBERTa (M4 Pro Optimized) 評估總結")
        print("="*80)
        summary_df = pd.DataFrame([{
            'Model': 'XLM-RoBERTa (M4 Pro)',
            'CV Avg Acc': avg_cv_acc,
            'Test Acc': acc,
            'F1-score': f1,
            'Final Train Time(s)': final_train_time
        }])
        print(summary_df.to_string(float_format="{:.4f}".format, index=False))
        
        print("\n[ 混淆矩陣圖 ]")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=label_names, yticklabels=label_names)
        plt.title(f'XLM-RoBERTa Confusion Matrix\nTest Acc: {acc:.4f}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()

🚀 M4 Pro (MPS) 效能全開模式已啟動！
1. 讀取與處理資料...
正在載入 Facebook AI 模型: xlm-roberta-base ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                                                                                

   -> Loss: 1.1129 | Acc: 0.6089
   Final Epoch 2/3


   -> Loss: 0.8203 | Acc: 0.7125
   Final Epoch 3/3


   -> Loss: 0.6769 | Acc: 0.7675

4. 最終測試集評估...


Testing:  54%|█████████████████▎              | 160/296 [00:58<00:49,  2.75it/s]